# Hierarchical Neighborhood Sampling

Scalable GNN Training on Reddit / Flickr: Layer-wise hierarchical mini-batch sampling for large graphs. This notebook implements the approach with `NeighborLoader` inside a `K3SampledGNN` model, trained with the Adam optimizer for 50 epochs, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `NeighborLoader` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install git+http://github.com/anas-rz/k3-node/@main

# ==============================================================================
# Part 2: K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops

import k3_node
from k3_node import layers as k3_layers
from k3_node.datasets import Planetoid

title = "Hierarchical Neighborhood Sampling Pipeline"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. Dataset
dataset = Planetoid(root="./data/Planetoid", name="Cora")
data = dataset[0]

# 2. Multi-Hop Sampling GNN
class K3SampledGNN(keras.Model):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = k3_layers.SAGEConv(in_channels, hidden_channels)
        self.conv2 = k3_layers.SAGEConv(hidden_channels, out_channels)

    def call(self, inputs, edge_index=None):
        if isinstance(inputs, (tuple, list)):
            x, edge_index = inputs[0], inputs[1]
        else:
            x = inputs
        x = ops.relu(self.conv1(x, edge_index))
        return self.conv2(x, edge_index)

k3_model = K3SampledGNN(dataset.num_features, 64, dataset.num_classes)

# 3. Model Compilation
k3_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
)

# 4. Training
def graph_data_generator():
    x = ops.convert_to_tensor(data.x, dtype="float32")
    edge_index = ops.convert_to_tensor(data.edge_index, dtype="int64")
    y = ops.convert_to_tensor(data.y, dtype="int64")
    mask = ops.cast(data.train_mask, "float32")
    while True:
        yield (x, edge_index), y, mask

print(f"Training K3-Node model on {backend} backend...")
k3_model.fit(graph_data_generator(), steps_per_epoch=1, epochs=5, verbose=1)

print("\n✓ K3-Node hierarchical sampling execution completed successfully!")
